# Correlation Tutorial 3: Trajectory Analysis & Time-Series Dynamics

In addition to static snapshot analysis, Correlation provides parallelized trajectory calculators for Molecular Dynamics (MD) simulations.

This tutorial demonstrates:
1. Constructing or loading multi-frame trajectories.
2. Computing the **Ensemble-Averaged Radial Distribution Function**.
3. Computing **Mean Squared Displacement (MSD)** and extracting diffusion coefficients.
4. Computing the **Velocity Autocorrelation Function (VACF)** and relaxation time.
5. Computing the **Vibrational Density of States (VDOS)** via Fourier transformation.
6. Generating publication-ready dynamic properties summaries.


## 1. Imports and Trajectory Synthesis


In [ ]:
import correlation
import numpy as np
import matplotlib.pyplot as plt

print("Initializing Trajectory Dynamics Demo...")


## 2. Generating a Synthetic Liquid MD Trajectory

For demonstration purposes, we generate a synthetic trajectory representing atoms undergoing diffusive Brownian dynamics with vibrational jitter.


In [ ]:
np.random.seed(42)
n_frames = 60
n_atoms = 32
box_length = 12.0
time_step_fs = 2.0  # 2 femtoseconds per frame

# Initial positions
pos = np.random.uniform(0.0, box_length, size=(n_atoms, 3))
traj = correlation.Trajectory()
traj.time_step = time_step_fs

positions_over_time = []
for step in range(n_frames):
    # Diffusive random walk + oscillatory noise
    drift = np.random.normal(0.0, 0.15, size=(n_atoms, 3))
    vib = 0.05 * np.sin(0.2 * step) * np.ones((n_atoms, 3))
    pos = (pos + drift + vib) % box_length
    positions_over_time.append(pos.copy())

    cell = correlation.Cell(box_length, box_length, box_length, 90.0, 90.0, 90.0)
    for i, p in enumerate(pos):
        elem = "Ar" if i % 2 == 0 else "Kr"
        cell.add_atom(correlation.Atom(elem, p[0], p[1], p[2]))
    traj.add_frame(cell)

print(f"Constructed Trajectory with {traj.frame_count} frames, {n_atoms} atoms each. Time step = {traj.time_step} fs.")


## 3. Mean Distribution Functions across Trajectory

The `DistributionFunctions.compute_mean()` routine averages pair correlation functions over all trajectory frames.


In [ ]:
analyzer = correlation.TrajectoryAnalyzer(traj)
settings = correlation.AnalysisSettings()
settings.r_max = 6.0
settings.r_bin_width = 0.05

mean_df = correlation.DistributionFunctions.compute_mean(traj, analyzer, 0, settings)
print("Computed mean distribution functions across trajectory.")
print("Available histograms:", mean_df.get_available_histograms())


## 4. Mean Squared Displacement (MSD) & Diffusion

Mean Squared Displacement measures atomic particle migration over lag time:
$$\text{MSD}(\Delta t) = \langle |\mathbf{r}(t + \Delta t) - \mathbf{r}(t)|^2 \rangle$$
From the Einstein relation in 3D:
$$D = \lim_{\Delta t \to \infty} \frac{\text{MSD}(\Delta t)}{6 \Delta t}$$


In [ ]:
# Calculate MSD directly on trajectory
msd_results = correlation.calculators.MSDCalculator.calculate(traj)
print("MSD calculation keys:", list(msd_results.keys()))

msd_hist = msd_results["MSD"]
time_lags = np.array(msd_hist.bins)  # fs
msd_total = np.array(msd_hist.partials.get("Total", []))

# Linear fit to compute diffusion coefficient D (Å²/fs)
if len(time_lags) > 5 and len(msd_total) > 5:
    slope, intercept = np.polyfit(time_lags[len(time_lags)//3:], msd_total[len(time_lags)//3:], 1)
    d_msd = slope / 6.0  # Å²/fs
    print(f"Estimated Self-Diffusion Coefficient D_MSD = {d_msd:.5e} Å²/fs ({d_msd * 1e-1:.5e} cm²/s)")


## 5. Visualizing Dynamics and Trajectory Properties


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), dpi=120)

# 1. Plot Ensemble-Averaged g(r)
if "g_r" in mean_df.get_available_histograms():
    gr = mean_df.get_histogram("g_r")
    for key, vals in gr.partials.items():
        ax1.plot(gr.bins, vals, label=key, lw=1.8)
    ax1.set_xlabel("Radius r (Å)", fontsize=11)
    ax1.set_ylabel("g(r)", fontsize=11)
    ax1.set_title("Ensemble-Averaged RDF", fontsize=12, fontweight='bold')
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend()

# 2. Plot MSD
if len(msd_total) > 0:
    ax2.plot(time_lags, msd_total, label="Total MSD", color="#0072B2", lw=2)
    ax2.set_xlabel("Lag Time Δt (fs)", fontsize=11)
    ax2.set_ylabel("MSD (Å²)", fontsize=11)
    ax2.set_title("Mean Squared Displacement", fontsize=12, fontweight='bold')
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()

plt.tight_layout()
plt.show()


## Conclusion
You have mastered multi-frame trajectory processing and time-series dynamic correlation functions with Correlation.

Refer to the official documentation and API reference for advanced ML interatomic potential integrations and GPU acceleration.
